# NLP Assignment 1 — Tokenization & Corpus Statistics

**Language:** Telugu (`te` / `tel_Telu`)  
**Corpora:** `ai4bharat/IndicCorpV2` (Task 1) and `oscar-corpus/OSCAR-2301` (Task 2)

<!-- This single notebook covers all four sub-tasks for **both** corpora:

| Task | Where it happens |
|---|---|
| **(a)** Download / stream the Telugu data | `stream_indiccorp()`, `stream_oscar()` |
| **(b)** Sentence tokenizer + word tokenizer (punctuation, URLs, e-mails, numbers, decimals, dates) | `sentence_tokenize()`, `word_tokenize()` |
| **(c)** Save tokenized data (one tokenized sentence per line, space-joined tokens) as compressed **Parquet** shards | `ShardedParquetWriter` |
| **(d)** Corpus statistics (sentences, words, characters, avg sentence length, avg word length, TTR) | `CorpusStats` |

### What was fixed relative to the earlier drafts (`NLP1.ipynb`, old `Final_Tokenizer_Stats.ipynb`)

1. **TTR was computed incorrectly.** The old code built a vocabulary of *word* tokens only, then divided it by the *all-token* count in the "ALL TOKENS" column. Two separate vocabularies are now tracked (`token_types` and `word_types`) so each column is internally consistent.
2. **Character counts were ambiguous.** Now reported explicitly as three numbers: characters over all tokens, over word tokens only, and over the cleaned raw text.
3. **Punctuation detection.** The old code re-matched every token against a `pure_punct` regex. The tokenizer already labels each token with its regex group, so the label is used directly — faster and consistent with the tokenizer.
4. **Word regex was Telugu/ASCII-only** (`[a-zA-Z\u0C00-\u0C7F]`), so it would silently shred any other script — a real problem for OSCAR. It is now built from Unicode letter + combining-mark classes, so it works for any language.
5. **Abbreviations were protected for sentence splitting but then shredded by the word tokenizer** (`కి.మీ.` → `కి . మీ .`). Abbreviations are now a first-class token group.
6. **Sentence splitter was O(boundaries × protected spans)** (a linear scan over every protected span for every candidate boundary). Replaced with an O(n) bitmask.
7. **Single Parquet file, no shards, writer never closed on error.** Now sharded output with `try/finally`, plus an optional plain-text mirror and a `*_stats.json` dump.
8. **Only IndicCorpV2 was handled.** The pipeline is now corpus-agnostic and OSCAR-2301 is included.
9. Extra token classes added: **times** (`10:30`), Indian **comma numbers with decimals** (`12,34,567.89`), hyphenated/apostrophe words, Devanagari danda (`।`) as a sentence end. -->

In [1]:
# If anything is missing, uncomment:
# !pip install datasets pyarrow tqdm -q

import os
import re
import json
import time
import unicodedata

import pyarrow as pa
import pyarrow.parquet as pq

print("pyarrow", pa.__version__)

pyarrow 25.0.0


## 0. Configuration

Set `MAX_RECORDS = None` to process the **entire** corpus (task (a) says *extract all the data*). Keep a number here while developing.

In [2]:
CONFIG = {
    # --- what to process -------------------------------------------------
    "INDICCORP_SPLIT":  "tel_Telu",   # Telugu split of IndicCorpV2
    "OSCAR_LANGUAGE":   "te",         # Telugu subset of OSCAR-2301
    "MAX_RECORDS":      None,       # <-- set to None for the FULL corpus
    "HF_TOKEN":         os.environ.get("HF_TOKEN"),  # OSCAR-2301 is a GATED dataset

    # --- where output goes ----------------------------------------------
    "OUT_DIR":          "data/output",
    "ROWS_PER_SHARD":   1_000_000,    # sentences per parquet shard
    "COMPRESSION":      "snappy",     # try "zstd" for ~30-40% smaller files
    "WRITE_TXT_MIRROR": False,        # True also writes plain .txt (much bigger)
    "PROGRESS_EVERY":   25_000,       # print progress every N paragraphs
}

os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
CONFIG

{'INDICCORP_SPLIT': 'tel_Telu',
 'OSCAR_LANGUAGE': 'te',
 'MAX_RECORDS': None,
 'HF_TOKEN': None,
 'OUT_DIR': 'data/output',
 'ROWS_PER_SHARD': 1000000,
 'COMPRESSION': 'snappy',
 'WRITE_TXT_MIRROR': False,
 'PROGRESS_EVERY': 25000}

In [ ]:
# ==============================================================================
# Unicode-general letter class: letters + combining marks (Mn/Mc/Me) + ZWJ/ZWNJ.
# Python's `re` has no \p{L}/\p{M}, and plain \w EXCLUDES Telugu vowel signs
# (they are category Mn), which would split "తెలంగాణలో" into pieces.
# So the mark ranges are enumerated once from the Unicode database.
# ==============================================================================
def _esc(cp):
    return "\\u%04x" % cp if cp <= 0xFFFF else "\\U%08x" % cp

def _combining_mark_class():
    ranges, start, prev = [], None, None
    for cp in range(0x0300, 0x1E020):
        if unicodedata.category(chr(cp)) in ("Mn", "Mc", "Me"):
            if start is None:
                start = cp
            prev = cp
        elif start is not None:
            ranges.append((start, prev))
            start = None
    if start is not None:
        ranges.append((start, prev))
    return "".join(_esc(a) if a == b else _esc(a) + "-" + _esc(b) for a, b in ranges)

MARKS  = _combining_mark_class()
ZWJ    = r'\u200c\u200d'                              # ZWNJ / ZWJ
LETTER = r'(?:[^\W\d_]|[' + MARKS + ZWJ + r'])'       # one "letter-ish" character

ABBREVIATIONS = [
    # Telugu
    "క్రీ.శ.", "క్రీ.పూ.", "చ.కి.మీ.", "కి.మీ.", "కి. మీ.", "మి.మీ.", "సె.మీ.",
    "డా.", "ప్రొ.", "శ్రీ.", "శ్రీమతి.", "నం.", "సం.", "తే.",
    # English
    "Dr.", "Mr.", "Mrs.", "Ms.", "Prof.", "Sr.", "Jr.", "St.", "No.", "Fig.",
    "vs.", "etc.", "e.g.", "i.e.", "U.S.", "U.K.", "A.M.", "P.M.",
]
_ABBR_ALT = "|".join(re.escape(a) for a in sorted(ABBREVIATIONS, key=len, reverse=True))

REGEX = re.compile(
    r'(?P<abbrev>' + _ABBR_ALT + r')'                          # known abbreviations
    r'|(?P<url>(?:https?://|www\.)[^\s<>"\']+)'                # URLs
    r'|(?P<email>[\w.+-]+@[\w-]+\.[\w.-]*\w)'                  # e-mails
    r'|(?P<date>\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4})'              # dates
    r'|(?P<time>\d{1,2}:\d{2}(?::\d{2})?)'                     # times
    r'|(?P<comma_num>\d{1,3}(?:,\d{2,3})+(?:\.\d+)?)'          # 1,00,000 / 12,34,567.89
    r'|(?P<decimal>\d+\.\d+)'                                  # 9.80
    r'|(?P<num_suffix>\d+(?:' + LETTER + r')+)'                # 2న, 18వ
    r'|(?P<number>\d+)'                                        # 2011
    r'|(?P<word>' + LETTER + r'+(?:[\'\u2019\-]' + LETTER + r'+)*)'   # words
    r'|(?P<punct>[^\s])'                                       # everything else
)

# Spans that must never be cut by the sentence splitter
PROTECTED_GROUPS = {"abbrev", "url", "email", "date", "time", "decimal", "comma_num", "num_suffix"}
# Token groups that are NOT words (used for the "WORDS ONLY" statistics column)
NON_WORD_GROUPS  = {"punct"}

SENTENCE_END  = re.compile(r'[.?!\u2026\u0964\u0965]+')   # . ? ! … । ॥
CLOSING_CHARS = '"\'\u201d\u2019)]}\u00bb'

DIGIT_MAP    = str.maketrans("౦౧౨౩౪౫౬౭౮౯", "0123456789")   # Telugu -> Arabic digits
WS_RE        = re.compile(r'\s+')
REP_PUNCT_RE = re.compile(r'([.!?\u2026])\1{1,}')

print("Master regex compiled. Groups:", list(REGEX.groupindex))

Master regex compiled. Groups: ['abbrev', 'url', 'email', 'date', 'time', 'comma_num', 'decimal', 'num_suffix', 'number', 'word', 'punct']


In [ ]:
def preprocess_text(text: str) -> str:
    """
    Cleaning applied to every raw paragraph:
      1. NFC Unicode normalisation (so identical strings compare equal)
      2. collapse all whitespace runs to a single space
      3. transliterate Telugu digits to Arabic digits
      4. collapse repeated sentence-final punctuation ("...", "!!!" -> ".", "!")
    """
    text = unicodedata.normalize('NFC', text)
    text = WS_RE.sub(' ', text).strip()
    text = text.translate(DIGIT_MAP)
    return REP_PUNCT_RE.sub(r'\1', text)


def word_tokenize(text: str, with_labels: bool = False):
    """Tokenize a sentence. with_labels=True returns (token, group_name) pairs."""
    if with_labels:
        return [(m.group(0), m.lastgroup) for m in REGEX.finditer(text)]
    return [m.group(0) for m in REGEX.finditer(text)]


def sentence_tokenize(text: str):
    """Split a preprocessed paragraph into sentences."""
    if not text:
        return []
    n = len(text)

    # O(n) bitmask of protected character positions (abbreviations, URLs, dates, ...)
    protected = bytearray(n + 1)
    for m in REGEX.finditer(text):
        if m.lastgroup in PROTECTED_GROUPS:
            for i in range(m.start() + 1, m.end() + 1):
                protected[i] = 1

    sentences, start = [], 0
    for m in SENTENCE_END.finditer(text):
        pos = m.end()
        if protected[pos]:            # e.g. the dot inside "కి.మీ." or "9.80"
            continue
        while pos < n and text[pos] in CLOSING_CHARS:   # keep trailing " ) ” with the sentence
            pos += 1
        chunk = text[start:pos].strip()
        if chunk:
            sentences.append(chunk)
        start = pos

    tail = text[start:].strip()
    if tail:
        sentences.append(tail)
    return sentences


def tokenize_paragraph(paragraph: str):
    """Full task-(b) pipeline: raw paragraph -> list of tokenized-sentence strings."""
    out = []
    for sent in sentence_tokenize(preprocess_text(paragraph)):
        toks = word_tokenize(sent)
        if toks:
            out.append(" ".join(toks))
    return out

In [5]:
# ------------------------------------------------------------------
# Tokenizer sanity tests — covers punctuation, URLs, e-mails,
# numbers, decimals, dates, times, abbreviations, quotes, Telugu digits
# ------------------------------------------------------------------
TESTS = [
    "ఇది కూడా చదవండి: Gold rate 8-5-2021: 1,00,000 పెరిగిన బంగారం ధర 9.80 లక్షలు",
    "వివరాలకు https://www.eenadu.net/news?id=12 చూడండి లేదా info@example.co.in కు మెయిల్ చేయండి. ధన్యవాదాలు!",
    "సబ్ పోస్టాఫీసు గ్రామానికి 5 కి.మీ. లోపు ఉంది. Dr. రావు 10.5 కి.మీ.కి పైబడి. సమయం 10:30కు.",
    "నమస్తే తెలంగాణ ప్రత్యేక కథనం… ఇది కొనసాగుతుంది.. ఏప్రిల్ ౨న?!",
    "\u201cశ్రీ నెట్టికంటి ఆంజనేయ స్వామి\u201d వారు కొలువై వున్నారు. Total 12,34,567.89 rupees on 01/02/2023.",
]

for raw in TESTS:
    clean = preprocess_text(raw)
    print("PARAGRAPH:", clean)
    for s in sentence_tokenize(clean):
        print("   SENT:", s)
        print("   TOKS:", " | ".join(f"{t}<{g}>" for t, g in word_tokenize(s, with_labels=True)))
    print("-" * 100)

PARAGRAPH: ఇది కూడా చదవండి: Gold rate 8-5-2021: 1,00,000 పెరిగిన బంగారం ధర 9.80 లక్షలు
   SENT: ఇది కూడా చదవండి: Gold rate 8-5-2021: 1,00,000 పెరిగిన బంగారం ధర 9.80 లక్షలు
   TOKS: ఇది<word> | కూడా<word> | చదవండి<word> | :<punct> | Gold<word> | rate<word> | 8-5-2021<date> | :<punct> | 1,00,000<comma_num> | పెరిగిన<word> | బంగారం<word> | ధర<word> | 9.80<decimal> | లక్షలు<word>
----------------------------------------------------------------------------------------------------
PARAGRAPH: వివరాలకు https://www.eenadu.net/news?id=12 చూడండి లేదా info@example.co.in కు మెయిల్ చేయండి. ధన్యవాదాలు!
   SENT: వివరాలకు https://www.eenadu.net/news?id=12 చూడండి లేదా info@example.co.in కు మెయిల్ చేయండి.
   TOKS: వివరాలకు<word> | https://www.eenadu.net/news?id=12<url> | చూడండి<word> | లేదా<word> | info@example.co.in<email> | కు<word> | మెయిల్<word> | చేయండి<word> | .<punct>
   SENT: ధన్యవాదాలు!
   TOKS: ధన్యవాదాలు<word> | !<punct>
-------------------------------------------------------------------------

In [6]:
class CorpusStats:
    """Incremental corpus statistics accumulator."""

    def __init__(self, name, track_vocab=True):
        self.name = name
        self.track_vocab = track_vocab
        self.paragraphs = 0
        self.sentences  = 0
        self.tokens     = 0      # all tokens, punctuation included
        self.words      = 0      # punctuation excluded
        self.token_chars = 0     # characters summed over all tokens
        self.word_chars  = 0     # characters summed over word tokens only
        self.raw_chars   = 0     # characters of the cleaned paragraph text (spaces included)
        self.token_types = set()
        self.word_types  = set()

    def add_paragraph(self, cleaned_text):
        self.paragraphs += 1
        self.raw_chars  += len(cleaned_text)

    def add_sentence(self, labeled_tokens):
        self.sentences += 1
        for tok, label in labeled_tokens:
            length = len(tok)
            self.tokens      += 1
            self.token_chars += length
            if self.track_vocab:
                self.token_types.add(tok)
            if label not in NON_WORD_GROUPS:
                self.words      += 1
                self.word_chars += length
                if self.track_vocab:
                    self.word_types.add(tok)

    @staticmethod
    def _div(a, b):
        return (a / b) if b else 0.0

    def as_dict(self):
        return {
            "corpus": self.name,
            "total_paragraphs": self.paragraphs,
            "total_sentences": self.sentences,
            "total_tokens_all": self.tokens,
            "total_words_only": self.words,
            "total_characters_all_tokens": self.token_chars,
            "total_characters_words_only": self.word_chars,
            "total_characters_raw_text": self.raw_chars,
            "avg_sentence_length_tokens": round(self._div(self.tokens, self.sentences), 4),
            "avg_sentence_length_words": round(self._div(self.words, self.sentences), 4),
            "avg_token_length": round(self._div(self.token_chars, self.tokens), 4),
            "avg_word_length": round(self._div(self.word_chars, self.words), 4),
            "unique_token_types": len(self.token_types),
            "unique_word_types": len(self.word_types),
            "ttr_all_tokens": round(self._div(len(self.token_types), self.tokens), 6),
            "ttr_words_only": round(self._div(len(self.word_types), self.words), 6),
        }


def print_stats_table(d):
    line = "=" * 74
    print(line)
    print(f"CORPUS STATISTICS: {d['corpus']}")
    print(line)
    print(f"{'Metric':<42} | {'ALL TOKENS':>13} | {'WORDS ONLY':>13}")
    print("-" * 74)
    rows = [
        ("i.   Total number of sentences",      d["total_sentences"],            d["total_sentences"]),
        ("ii.  Total number of words/tokens",   d["total_tokens_all"],           d["total_words_only"]),
        ("iii. Total number of characters",     d["total_characters_all_tokens"], d["total_characters_words_only"]),
    ]
    for label, a, b in rows:
        print(f"{label:<42} | {a:>13,} | {b:>13,}")
    print(f"{'iv.  Average sentence length':<42} | {d['avg_sentence_length_tokens']:>13.2f} | {d['avg_sentence_length_words']:>13.2f}")
    print(f"{'v.   Average word length':<42} | {d['avg_token_length']:>13.2f} | {d['avg_word_length']:>13.2f}")
    print(f"{'vi.  Type/Token Ratio (TTR)':<42} | {d['ttr_all_tokens']:>13.4f} | {d['ttr_words_only']:>13.4f}")
    print("-" * 74)
    print(f"{'     Unique types (vocabulary)':<42} | {d['unique_token_types']:>13,} | {d['unique_word_types']:>13,}")
    print(f"{'     Paragraphs processed':<42} | {d['total_paragraphs']:>13,} |")
    print(f"{'     Characters in cleaned raw text':<42} | {d['total_characters_raw_text']:>13,} |")
    if "elapsed_seconds" in d:
        print(f"{'     Elapsed (s)':<42} | {d['elapsed_seconds']:>13,.1f} |")
    print(line)

In [7]:
class ShardedParquetWriter:
    """Writes tokenized sentences (one per row) into compressed, sharded Parquet files."""

    def __init__(self, out_dir, prefix, rows_per_shard=1_000_000, batch_rows=50_000,
                 compression="snappy", also_txt=False):
        os.makedirs(out_dir, exist_ok=True)
        self.out_dir     = out_dir
        self.prefix      = prefix
        self.rows_per_shard = rows_per_shard
        self.batch_rows  = min(batch_rows, rows_per_shard)
        self.compression = compression
        self.also_txt    = also_txt
        self.schema      = pa.schema([("tokenized_sentence", pa.string())])
        self._buf, self._writer, self._txt = [], None, None
        self.shard_idx = self.rows_in_shard = self.total_rows = 0
        self.files = []

    def _open_shard(self):
        self.shard_idx += 1
        path = os.path.join(self.out_dir, f"{self.prefix}_part-{self.shard_idx:05d}.parquet")
        self._writer = pq.ParquetWriter(path, self.schema, compression=self.compression)
        self.files.append(path)
        self.rows_in_shard = 0
        if self.also_txt:
            self._txt = open(path.replace(".parquet", ".txt"), "w", encoding="utf-8")

    def write(self, line):
        self._buf.append(line)
        if len(self._buf) >= self.batch_rows:
            self._flush()

    def _flush(self):
        if not self._buf:
            return
        if self._writer is None:
            self._open_shard()
        table = pa.Table.from_arrays([pa.array(self._buf, type=pa.string())], schema=self.schema)
        self._writer.write_table(table)
        if self._txt:
            self._txt.write("\n".join(self._buf) + "\n")
        self.rows_in_shard += len(self._buf)
        self.total_rows    += len(self._buf)
        self._buf = []
        if self.rows_in_shard >= self.rows_per_shard:
            self._close_shard()

    def _close_shard(self):
        if self._writer is not None:
            self._writer.close()
            self._writer = None
        if self._txt is not None:
            self._txt.close()
            self._txt = None

    def close(self):
        self._flush()
        self._close_shard()

In [8]:
def process_corpus(text_iter, corpus_name, prefix, max_records=None, track_vocab=True):
    """
    Stream raw paragraphs -> clean -> sentence split -> word tokenize
    -> write '<tok> <tok> ...' rows to sharded parquet -> accumulate statistics.

    Returns the statistics dict (also written to <prefix>_stats.json).
    """
    out_dir = CONFIG["OUT_DIR"]
    stats  = CorpusStats(corpus_name, track_vocab=track_vocab)
    writer = ShardedParquetWriter(
        out_dir, prefix,
        rows_per_shard=CONFIG["ROWS_PER_SHARD"],
        compression=CONFIG["COMPRESSION"],
        also_txt=CONFIG["WRITE_TXT_MIRROR"],
    )
    every = CONFIG["PROGRESS_EVERY"]
    t0 = time.time()
    seen = 0

    print(f"Processing {corpus_name} ...")
    try:
        for raw in text_iter:
            if max_records is not None and seen >= max_records:
                break
            seen += 1
            if not raw or not raw.strip():
                continue                      # IndicCorpV2 contains many blank records

            text = preprocess_text(raw)
            if not text:
                continue
            stats.add_paragraph(text)

            for sent in sentence_tokenize(text):
                labeled = word_tokenize(sent, with_labels=True)
                if not labeled:
                    continue
                stats.add_sentence(labeled)
                writer.write(" ".join(tok for tok, _ in labeled))

            if every and stats.paragraphs % every == 0:
                print(f"  {stats.paragraphs:,} paragraphs | {stats.sentences:,} sentences "
                      f"| {time.time() - t0:.1f}s", flush=True)
    finally:
        writer.close()                        # never leave a half-written parquet file

    d = stats.as_dict()
    d["elapsed_seconds"] = round(time.time() - t0, 2)
    d["output_rows"]     = writer.total_rows
    d["output_files"]    = writer.files
    d["compression"]     = CONFIG["COMPRESSION"]

    with open(os.path.join(out_dir, f"{prefix}_stats.json"), "w", encoding="utf-8") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

    print(f"Done: {writer.total_rows:,} tokenized sentences -> {len(writer.files)} parquet shard(s)")
    return d

In [9]:
from datasets import load_dataset


def stream_indiccorp(split=None):
    """Yield raw paragraph strings from IndicCorpV2."""
    split = split or CONFIG["INDICCORP_SPLIT"]
    ds = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split=split, streaming=True)
    for ex in ds:
        yield ex["text"]


def stream_oscar(language=None, token=None):
    """Yield raw document strings from OSCAR-2301 (gated dataset -> needs a token)."""
    language = language or CONFIG["OSCAR_LANGUAGE"]
    token    = token or CONFIG["HF_TOKEN"] or True   # True = use the cached CLI login
    ds = load_dataset(
        "oscar-corpus/OSCAR-2301",
        language=language,
        split="train",
        streaming=True,
        token=token,
        trust_remote_code=True,
    )
    for ex in ds:
        yield ex["text"]

In [1]:
# Peek at a couple of records before the full run (cheap sanity check)
peek = stream_indiccorp()
for i, t in enumerate(peek):
    if i >= 3:
        break
    print(f"[{i}] {t[:200]}\n")
del peek

NameError: name 'stream_indiccorp' is not defined

## 6. Run — Corpus 1: IndicCorpV2 (Telugu)

In [16]:
stats_indic = process_corpus(
    stream_indiccorp(),
    corpus_name=f"IndicCorpV2 ({CONFIG['INDICCORP_SPLIT']})",
    prefix="indiccorp_te_tokenized",
    max_records=CONFIG["MAX_RECORDS"],
)
print_stats_table(stats_indic)

Processing IndicCorpV2 (tel_Telu) ...
  25,000 paragraphs | 97,423 sentences | 77.7s
  50,000 paragraphs | 193,759 sentences | 104.6s
  75,000 paragraphs | 290,127 sentences | 110.6s
  100,000 paragraphs | 385,082 sentences | 153.6s
  125,000 paragraphs | 479,731 sentences | 175.2s
  150,000 paragraphs | 577,379 sentences | 180.1s
  175,000 paragraphs | 674,247 sentences | 200.6s
  200,000 paragraphs | 769,060 sentences | 221.9s
  225,000 paragraphs | 866,596 sentences | 226.7s
  250,000 paragraphs | 962,870 sentences | 246.7s
  275,000 paragraphs | 1,058,191 sentences | 268.0s
  300,000 paragraphs | 1,153,960 sentences | 272.9s
  325,000 paragraphs | 1,251,438 sentences | 295.6s
  350,000 paragraphs | 1,347,906 sentences | 314.3s
  375,000 paragraphs | 1,445,008 sentences | 319.1s
  400,000 paragraphs | 1,539,025 sentences | 334.7s
  425,000 paragraphs | 1,634,956 sentences | 348.9s
  450,000 paragraphs | 1,730,922 sentences | 353.6s
  475,000 paragraphs | 1,828,040 sentences | 366.9s

## 7. Run — Corpus 2: OSCAR-2301 (Telugu)

Run this only after accepting the dataset terms and logging in; otherwise it raises a gated-repo error.

In [17]:
try:
    stats_oscar = process_corpus(
        stream_oscar(),
        corpus_name=f"OSCAR-2301 ({CONFIG['OSCAR_LANGUAGE']})",
        prefix="oscar2301_te_tokenized",
        max_records=CONFIG["MAX_RECORDS"],
    )
    print_stats_table(stats_oscar)
except Exception as e:
    stats_oscar = None
    print("OSCAR run failed:", type(e).__name__, e)
    print("\nIf this is a 401/403 GatedRepoError: accept the terms at")
    print("https://huggingface.co/datasets/oscar-corpus/OSCAR-2301 and run `huggingface-cli login`,")
    print("or set CONFIG['HF_TOKEN'].")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'oscar-corpus/OSCAR-2301' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Processing OSCAR-2301 (te) ...
OSCAR run failed: LocalTokenNotFoundError Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

If this is a 401/403 GatedRepoError: accept the terms at
https://huggingface.co/datasets/oscar-corpus/OSCAR-2301 and run `huggingface-cli login`,
or set CONFIG['HF_TOKEN'].


## 8. Verify the saved files (read the Parquet back)

In [18]:
def inspect_output(prefix, n=5):
    files = sorted(f for f in os.listdir(CONFIG["OUT_DIR"])
                   if f.startswith(prefix) and f.endswith(".parquet"))
    if not files:
        print(f"No parquet files found for prefix '{prefix}'")
        return
    total_bytes = sum(os.path.getsize(os.path.join(CONFIG["OUT_DIR"], f)) for f in files)
    total_rows  = 0
    for f in files:
        pf = pq.ParquetFile(os.path.join(CONFIG["OUT_DIR"], f))
        total_rows += pf.metadata.num_rows
    print(f"{prefix}: {len(files)} shard(s), {total_rows:,} rows, {total_bytes/1024**2:.2f} MB on disk")

    pf = pq.ParquetFile(os.path.join(CONFIG["OUT_DIR"], files[0]))
    batch = next(pf.iter_batches(batch_size=n))          # lazy: does not load the whole file
    for row in batch.column("tokenized_sentence").to_pylist():
        print("   ", row[:160])
    print()

inspect_output("indiccorp_te_tokenized")
inspect_output("oscar2301_te_tokenized")

indiccorp_te_tokenized: 83 shard(s), 82,507,030 rows, 6303.39 MB on disk
    అమెరికా అధ్యక్షుడు డొనాల్డ్ ట్రంప్ కు రాష్ట్రపతి భవన్ వద్ద ఘనస్వాగతం లభించింది .
    ఆయనకు రాష్ట్రపతి రామ్ నాథ్ కోవింద్ దంపతులు , ప్రధాని మోదీ సాదరంగా ఆహ్వానం పలకడంతో పాటు సైనికులు గౌరవ వందనాన్ని అందించారు .
    ఇటు తెలంగాణలో కరోనా వైరస్ కారణంగా అన్ని దేవాలయాల్లో ముందస్తు చర్యలు చేపట్టారు .
    భద్రాద్రి రాముడికి కరోనా ఎఫెక్ట్ తగిలింది .
    ఏప్రిల్ 2న భద్రాద్రిలో జరగనున్న శ్రీరామ నవమి వేడుకలను వెంటాడుతోంది కరోనా .

No parquet files found for prefix 'oscar2301_te_tokenized'


## 9. Side-by-side comparison + README table

In [19]:
def markdown_table(*stat_dicts):
    stat_dicts = [d for d in stat_dicts if d]
    if not stat_dicts:
        return "(no statistics available)"
    rows = [
        ("i. Total sentences",                "total_sentences",             "{:,}"),
        ("ii. Total tokens (with punctuation)", "total_tokens_all",          "{:,}"),
        ("ii. Total words (no punctuation)",  "total_words_only",            "{:,}"),
        ("iii. Total characters (all tokens)", "total_characters_all_tokens", "{:,}"),
        ("iii. Total characters (words only)", "total_characters_words_only", "{:,}"),
        ("iv. Avg sentence length (words)",   "avg_sentence_length_words",   "{:.2f}"),
        ("v. Avg word length (chars)",        "avg_word_length",             "{:.2f}"),
        ("vi. TTR (all tokens)",              "ttr_all_tokens",              "{:.4f}"),
        ("vi. TTR (words only)",              "ttr_words_only",              "{:.4f}"),
        ("Unique word types",                 "unique_word_types",           "{:,}"),
        ("Paragraphs / documents",            "total_paragraphs",            "{:,}"),
    ]
    head = "| Metric | " + " | ".join(d["corpus"] for d in stat_dicts) + " |"
    sep  = "|---" * (len(stat_dicts) + 1) + "|"
    body = [head, sep]
    for label, key, fmt in rows:
        body.append("| " + label + " | " + " | ".join(fmt.format(d[key]) for d in stat_dicts) + " |")
    return "\n".join(body)


table = markdown_table(stats_indic, stats_oscar if "stats_oscar" in dir() else None)
print(table)

with open(os.path.join(CONFIG["OUT_DIR"], "corpus_statistics.md"), "w", encoding="utf-8") as f:
    f.write("# Corpus statistics\n\n" + table + "\n")
print("\nSaved -> data/output/corpus_statistics.md  (paste this into your README.md)")

| Metric | IndicCorpV2 (tel_Telu) |
|---|---|
| i. Total sentences | 82,507,030 |
| ii. Total tokens (with punctuation) | 863,456,797 |
| ii. Total words (no punctuation) | 735,129,680 |
| iii. Total characters (all tokens) | 5,125,034,281 |
| iii. Total characters (words only) | 4,996,707,164 |
| iv. Avg sentence length (words) | 8.91 |
| v. Avg word length (chars) | 6.80 |
| vi. TTR (all tokens) | 0.0114 |
| vi. TTR (words only) | 0.0134 |
| Unique word types | 9,853,523 |
| Paragraphs / documents | 21,458,261 |

Saved -> data/output/corpus_statistics.md  (paste this into your README.md)
